# Full Fine-Tuning: EN→FR pretraining then EN→DE fine-tuning

In [ ]:
import sacrebleu

import os
import gc
import re
import math
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm
import functools
from torch import optim
from torch.utils.data import TensorDataset, DataLoader, random_split, Subset
from torch.cuda.amp import GradScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from datasets import Dataset, load_from_disk
from torch.utils.data import DataLoader, random_split  
import pickle


SEED = 1070
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Defaulting to user installation because normal site-packages is not writeable
  Using cached sacrebleu-2.6.0-py3-none-any.whl.metadata (39 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached sacrebleu-2.6.0-py3-none-any.whl (100 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 4.4 MB/s eta 0:00:01
   ---------- ----------------------------- 1.0/4.0 MB 2.8 MB/s eta 0:00:02
   ------------------ --------------------- 1.8/4.0 MB 3.2 MB/s eta 0:00:01
   -------------------------- ------------- 2.6/4.0 MB 3.4 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.0 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 3.8 MB/s  0:00:01
Using cached portalocker-3.2.0-py3-none-any.whl (22 kB)

   -------- ------------------------

ModuleNotFoundError: No module named 'tokenizers'

In [ ]:
CFG = {
    "en_fr_path":            "/home/ubuntu/10701-project/en-fr.csv",
    "juridia_train":         "/home/ubuntu/10701-project/en-fr.csv",
    "pretrain_sample_frac":  0.001,
    "finetune_sample_frac":  0.001,
    "vocab_size":            32000,   # shared BPE vocab (src+tgt together)
    "max_len":               128,
    "d_model":               256,
    "n_heads":               8,
    "d_k":                   32,
    "d_v":                   32,
    "d_ff":                  512,
    "n_layers":              4,
    "batch_size":            256,
    "lr_pretrain":           3e-4,
    "lr_finetune":           5e-4,
    "pretrain_epochs":       1,
    "finetune_epochs":       2,
    "train_split":           0.95,
    "num_workers":           4 if torch.cuda.is_available() else 0,
    "lora_r":                8,
    "lora_alpha":            16,
    "lora_dropout":          0.1,
    "adapter_bottleneck":    64,
    "adapter_dropout":       0.1,
    "lr_adapter":            5e-4,
    "adapter_epochs":        2,
    "tokenizer_path":        "/home/ubuntu/10701-project/bpe_tokenizer.json",
    "cache_dir":             "/home/ubuntu/10701-project/hf_cache",
    "processed_cache_dir":   "/home/ubuntu/10701-project/processed_cache",
    "cache_version":         "v1",
}
CFG["grad_accum_steps"] = max(1, CFG["batch_size"] // CFG["batch_size"])
Path(CFG["processed_cache_dir"]).mkdir(parents=True, exist_ok=True)



CACHE_FILE = "/home/ubuntu/10701-project/preprocessed_data_cache2.pkl"

In [ ]:
gc.collect()
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'([^\w\s])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_csv_as_src_tgt(path, src_col, tgt_col, sample_frac=1.0):
    df = pd.read_csv(path, usecols=[src_col, tgt_col])
    df = df.dropna().drop_duplicates().reset_index(drop=True)
    if sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=SEED).reset_index(drop=True)
    df[src_col] = df[src_col].apply(clean_text)
    df[tgt_col] = df[tgt_col].apply(clean_text)
    return df.rename(columns={src_col: "src", tgt_col: "tgt"})

def cache_name(*parts):
    safe = "_".join(str(p).replace("/", "-").replace(" ", "") for p in parts)
    return Path(CFG["processed_cache_dir"]) / safe

def load_or_build_clean_splits():
    cache_path = cache_name(
        "clean_splits",
        CFG["cache_version"],
        f"pre{CFG['pretrain_sample_frac']}",
        f"ft{CFG['finetune_sample_frac']}",
        f"seed{SEED}",
    ).with_suffix(".pkl")

    if cache_path.exists():
        print(f"Loading cleaned data splits from {cache_path}")
        cached = pd.read_pickle(cache_path)
        return cached["pretrain_df"], cached["finetune_train_df"], cached["finetune_test_df"], cached["ft_df"]

    print("Loading pretrain data...")
    pretrain_df = load_csv_as_src_tgt(
        CFG["en_fr_path"], "fr", "en", CFG["pretrain_sample_frac"])
    print("Loading finetune data...")
    ft_df = load_csv_as_src_tgt(
        CFG["juridia_train"], "fr", "en", CFG["finetune_sample_frac"])

    print("Finished loading finetune data")
    finetune_train_df, finetune_test_df = train_test_split(
        ft_df, test_size=0.1, random_state=SEED, shuffle=True)
    finetune_train_df = finetune_train_df.reset_index(drop=True)
    finetune_test_df  = finetune_test_df.reset_index(drop=True)

    pd.to_pickle({
        "pretrain_df": pretrain_df,
        "finetune_train_df": finetune_train_df,
        "finetune_test_df": finetune_test_df,
        "ft_df": ft_df,
    }, cache_path)
    print(f"Saved cleaned data splits to {cache_path}")
    return pretrain_df, finetune_train_df, finetune_test_df, ft_df
if not os.path.exists(CACHE_FILE):
    pretrain_df, finetune_train_df, finetune_test_df, ft_df = load_or_build_clean_splits()
    print(f"Pretrain: {pretrain_df.shape}  |  FT train: {finetune_train_df.shape}  |  FT test: {finetune_test_df.shape}")


In [ ]:

def sentence_iterator(dfs, cols): 
    for df in dfs: 
        for col in cols: 
            for text in df[col]:
                yield str(text)

def train_or_load_tokenizer(cfg, dfs) -> Tokenizer:
    if os.path.exists(cfg["tokenizer_path"]):
        print(f"Loading tokenizer from {cfg['tokenizer_path']}")
        return Tokenizer.from_file(cfg["tokenizer_path"])

    print("Training BPE tokenizer...")
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=cfg["vocab_size"],
        special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"],
        min_frequency=2,
        show_progress=True,
    )
    tokenizer.train_from_iterator(
        sentence_iterator(dfs, ["src", "tgt"]),
        trainer=trainer,
        length=sum(len(df) * 2 for df in dfs),
    )

    bos_id = tokenizer.token_to_id("<bos>")
    eos_id = tokenizer.token_to_id("<eos>")
    tokenizer.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[("<bos>", bos_id), ("<eos>", eos_id)],
    )
    tokenizer.enable_padding(
        pad_id=tokenizer.token_to_id("<pad>"),
        pad_token="<pad>",
        length=cfg["max_len"],
    )
    tokenizer.enable_truncation(max_length=cfg["max_len"])

    tokenizer.save(cfg["tokenizer_path"])
    print(f"Tokenizer saved -> {cfg['tokenizer_path']}")
    return tokenizer
if not os.path.exists(CACHE_FILE):
    tokenizer = train_or_load_tokenizer(
    CFG, [pretrain_df, finetune_train_df])

    PAD_ID = tokenizer.token_to_id("<pad>")
    VOCAB_SIZE = tokenizer.get_vocab_size()
    print(f"Vocab size: {VOCAB_SIZE}  |  PAD id: {PAD_ID}")
gc.collect()


In [ ]:
if not os.path.exists(CACHE_FILE):
    tokenizer.no_padding()
    tokenizer.enable_truncation(max_length=CFG["max_len"])

    #collate
    def collate_fn_translation(batch, pad_id):
        src_list = [item["src_ids"] for item in batch]
        tgt_list = [item["tgt_ids"] for item in batch]
        max_src = max(s.size(0) for s in src_list)
        max_tgt = max(t.size(0) for t in tgt_list)
        src_padded = torch.full((len(batch), max_src), pad_id, dtype=torch.long)
        tgt_padded = torch.full((len(batch), max_tgt), pad_id, dtype=torch.long)
        for i, (src, tgt) in enumerate(zip(src_list, tgt_list)):
            src_padded[i, :src.size(0)] = src
            tgt_padded[i, :tgt.size(0)] = tgt
        return src_padded, tgt_padded


    _collate = functools.partial(collate_fn_translation, pad_id=PAD_ID)

    def tokenize_batch(examples):
        return {
            "src_ids": [tokenizer.encode(str(t)).ids for t in examples["src"]],
            "tgt_ids": [tokenizer.encode(str(t)).ids for t in examples["tgt"]],
        }

    #caching 
    def hf_cache_path(name, df):
        tokenizer_sig = f"vocab{CFG['vocab_size']}_len{CFG['max_len']}"
        data_sig = (
            f"n{len(df)}"
            f"_pre{CFG['pretrain_sample_frac']}"
            f"_ft{CFG['finetune_sample_frac']}"
            f"_seed{SEED}"
        )

        return str(cache_name("tokenized_hf", name, CFG["cache_version"], tokenizer_sig, data_sig))

    #Load from disk if cached, otherwise tokenize and save
    def load_or_tokenize_hf(name, df):
        path = hf_cache_path(name, df)
        if os.path.exists(path):
            print(f"Loading cached HF dataset for '{name}' from {path}")
            return load_from_disk(path)

        print(f"Tokenizing '{name}' data …")
        hf_ds = Dataset.from_pandas(df[["src", "tgt"]])
        hf_ds = hf_ds.map(
            tokenize_batch,
            batched=True,
            remove_columns=["src", "tgt"],
            desc=f"Tokenizing {name}",
        )
        hf_ds.save_to_disk(path)
        print(f"Saved tokenized '{name}' dataset to {path}")
        return hf_ds

    #  DataLoader builders
    def make_hf_loaders(hf_ds, train_split, batch_size, num_workers):
        split = hf_ds.train_test_split(train_size=train_split, seed=SEED)

        train_ds = split["train"].with_format(type="torch", columns=["src_ids", "tgt_ids"])
        val_ds   = split["test"].with_format(type="torch", columns=["src_ids", "tgt_ids"])

        kw = dict(
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            collate_fn=_collate,
        )
        train_loader = DataLoader(train_ds, batch_size=batch_size,shuffle=True,  drop_last=True,  **kw)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size,shuffle=False, drop_last=False, **kw)
        return train_loader, val_loader

    def make_hf_test_loader(hf_ds, batch_size, num_workers):
        ds = hf_ds.with_format(type="torch", columns=["src_ids", "tgt_ids"])
        return DataLoader(ds, batch_size=batch_size, shuffle=False,num_workers=num_workers,pin_memory=torch.cuda.is_available(),drop_last=False, collate_fn=_collate)


    pretrain_hf = load_or_tokenize_hf("pretrain",       pretrain_df)
    finetune_train_hf = load_or_tokenize_hf("finetune_train", finetune_train_df)
    finetune_test_hf = load_or_tokenize_hf("finetune_test",  finetune_test_df)


    pretrain_train_loader, pretrain_val_loader = make_hf_loaders(pretrain_hf, CFG["train_split"], CFG["batch_size"], CFG["num_workers"])
    finetune_train_loader, finetune_val_loader = make_hf_loaders(finetune_train_hf, CFG["train_split"], CFG["batch_size"], CFG["num_workers"])
    finetune_test_loader = make_hf_test_loader(finetune_test_hf, CFG["batch_size"], CFG["num_workers"])

    print(f"Pretrain  — train: {len(pretrain_train_loader)} | val: {len(pretrain_val_loader)}")
    print(f"Finetune  — train: {len(finetune_train_loader)} | val: {len(finetune_val_loader)}")
    print(f"Finetune  — test:  {len(finetune_test_loader)}")

    del pretrain_df, finetune_train_df, finetune_test_df, ft_df
    gc.collect()

In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
 
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]
 
 
class LoRALinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with a trainable low-rank delta.
    """
    def __init__(self, in_features: int, out_features: int,
                 r: int, lora_alpha: float, lora_dropout: float = 0.0,
                 bias: bool = True):
        super().__init__()
        assert r > 0, "LoRA rank r must be a positive integer"
        self.in_features  = in_features
        self.out_features = out_features
        self.r            = r
        self.scaling      = lora_alpha / r
 
        # Frozen base weight (same layout as nn.Linear)
        self.weight = nn.Parameter(torch.empty(out_features, in_features),
                                   requires_grad=False)
        self.bias   = nn.Parameter(torch.zeros(out_features),
                                   requires_grad=False) if bias else None
 
        #Trainable LoRA matrices
        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
 
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0.0 else nn.Identity()
        self._merged = False
        self._reset_base_weights()
 
    def _reset_base_weights(self):
        """Initialise W and b exactly as nn.Linear does."""
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)
        # A gets Kaiming so gradients flow from the very first step
        # B stays zero so the initial model output is identical to the pretrained one.
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base linear
        out = F.linear(x, self.weight, self.bias)
        if not self._merged:
            # LoRA delta: x → (r,) → (out,)
            lora_out = F.linear(self.lora_dropout(x), self.lora_A)
            lora_out = F.linear(lora_out, self.lora_B)
            out = out + lora_out * self.scaling
        return out
 
    def merge(self):
        """
        Fold the LoRA delta into W in-place.
        After merging, forward() is identical to a plain nn.Linear — zero overhead.
        """
        if self._merged:
            return
        # delta W = B @ A, shape (out, in) same as self.weight
        with torch.no_grad():
            self.weight.data += (self.lora_B @ self.lora_A) * self.scaling
        self._merged = True
 
    def extra_repr(self) -> str:
        return (f"in={self.in_features}, out={self.out_features}, "
                f"r={self.r}, scaling={self.scaling:.3f}, merged={self._merged}")
 
 
class AttentionLayer(nn.Module):
    """Single attention head. Optionally wraps Q/K/V projections with LoRA."""
    def __init__(self, d_model, d_k, d_v, lora_cfg=None):
        super().__init__()
        if lora_cfg:
            r, alpha, drop = lora_cfg["r"], lora_cfg["alpha"], lora_cfg["dropout"]
            self.W_q = LoRALinear(d_model, d_k, r=r, lora_alpha=alpha, lora_dropout=drop)
            self.W_k = LoRALinear(d_model, d_k, r=r, lora_alpha=alpha, lora_dropout=drop)
            self.W_v = LoRALinear(d_model, d_v, r=r, lora_alpha=alpha, lora_dropout=drop)
        else:
            self.W_q = nn.Linear(d_model, d_k)
            self.W_k = nn.Linear(d_model, d_k)
            self.W_v = nn.Linear(d_model, d_v)
 
    def forward(self, q, k, v, mask=None):
        Q, K, V = self.W_q(q), self.W_k(k), self.W_v(v)
        scores = (Q @ K.transpose(-1, -2)) / math.sqrt(K.size(-1))
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(0)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        return torch.softmax(scores, dim=-1) @ V
 
 
class AdapterLayer(nn.Module):
    """
    Houlsby-style bottleneck adapter (Houlsby et al., 2019).
    """
    def __init__(self, d_model: int, bottleneck: int, dropout: float = 0.0):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck)
        self.up      = nn.Linear(bottleneck, d_model)
        self.act     = nn.GELU()
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        self._init_weights()
 
    def _init_weights(self):
        # Small normal init for down
        nn.init.normal_(self.down.weight, std=0.01)
        nn.init.zeros_(self.down.bias)
        # Zero init for up
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.up(self.dropout(self.act(self.down(x))))
 
    def extra_repr(self) -> str:
        return f"d_model→{self.down.out_features}→{self.up.out_features}"
 
 
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, lora_cfg=None):
        super().__init__()
        self.heads = nn.ModuleList(
            [AttentionLayer(d_model, d_k, d_v, lora_cfg) for _ in range(n_heads)])
        self.lin = nn.Linear(n_heads * d_v, d_model)
 
    def forward(self, q, k, v, mask=None):
        return self.lin(torch.cat([h(q, k, v, mask) for h in self.heads], dim=-1))
 
 
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
 
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))
 
 
class EncoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, d_ff, lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.mha = MultiHeadAttention(n_heads, d_model, d_k, d_v, lora_cfg)
        self.ffn = FeedForward(d_model, d_ff)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        # Houlsby: one adapter after attention sub-layer, one after FFN sub-layer
        if adapter_cfg:
            b, drop = adapter_cfg["bottleneck"], adapter_cfg["dropout"]
            self.adapter_attn = AdapterLayer(d_model, b, drop)
            self.adapter_ffn  = AdapterLayer(d_model, b, drop)
        else:
            self.adapter_attn = None
            self.adapter_ffn  = None
 
    def forward(self, x, src_mask=None):
        x = self.ln1(x + self.mha(x, x, x, src_mask))
        if self.adapter_attn is not None:
            x = self.adapter_attn(x)
        x = self.ln2(x + self.ffn(x))
        if self.adapter_ffn is not None:
            x = self.adapter_ffn(x)
        return x
 
 
class DecoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_k, d_v, d_ff, lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.self_attn  = MultiHeadAttention(n_heads, d_model, d_k, d_v, lora_cfg)
        self.cross_attn = MultiHeadAttention(n_heads, d_model, d_k, d_v, lora_cfg)
        self.ffn  = FeedForward(d_model, d_ff)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.ln3  = nn.LayerNorm(d_model)
        # Houlsby: adapter after self-attn, after cross-attn, and after FFN
        if adapter_cfg:
            b, drop = adapter_cfg["bottleneck"], adapter_cfg["dropout"]
            self.adapter_self  = AdapterLayer(d_model, b, drop)
            self.adapter_cross = AdapterLayer(d_model, b, drop)
            self.adapter_ffn   = AdapterLayer(d_model, b, drop)
        else:
            self.adapter_self  = None
            self.adapter_cross = None
            self.adapter_ffn   = None
 
    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        tgt = self.ln1(tgt + self.self_attn(tgt, tgt, tgt, tgt_mask))
        if self.adapter_self is not None:
            tgt = self.adapter_self(tgt)
        tgt = self.ln2(tgt + self.cross_attn(tgt, memory, memory, src_mask))
        if self.adapter_cross is not None:
            tgt = self.adapter_cross(tgt)
        tgt = self.ln3(tgt + self.ffn(tgt))
        if self.adapter_ffn is not None:
            tgt = self.adapter_ffn(tgt)
        return tgt
 
 
class Encoder(nn.Module):
    def __init__(self, vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                 lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.layers  = nn.ModuleList(
            [EncoderLayer(n_heads, d_model, d_k, d_v, d_ff, lora_cfg, adapter_cfg)
             for _ in range(n_layers)])
 
    def forward(self, x, src_mask=None):
        x = self.pos_enc(self.embed(x))
        for layer in self.layers:
            x = layer(x, src_mask)
        return x
 
 
class Decoder(nn.Module):
    def __init__(self, vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                 lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        self.layers  = nn.ModuleList(
            [DecoderLayer(n_heads, d_model, d_k, d_v, d_ff, lora_cfg, adapter_cfg)
             for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)
 
    def forward(self, tgt, memory, tgt_mask=None, src_mask=None):
        x = self.pos_enc(self.embed(tgt))
        for layer in self.layers:
            x = layer(x, memory, tgt_mask, src_mask)
        return self.out(x)
 
 
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 n_heads, d_model, d_k, d_v, d_ff, n_layers,
                 lora_cfg=None, adapter_cfg=None):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                                lora_cfg, adapter_cfg)
        self.decoder = Decoder(tgt_vocab_size, n_heads, d_model, d_k, d_v, d_ff, n_layers,
                                lora_cfg, adapter_cfg)
 
    def _causal_mask(self, sz, dev):
        return ~torch.triu(torch.ones(sz, sz, device=dev, dtype=torch.bool), diagonal=1)
 
    def forward(self, src, tgt):
        tgt_mask = self._causal_mask(tgt.size(1), src.device)
        memory   = self.encoder(src)
        return self.decoder(tgt, memory, tgt_mask=tgt_mask)

In [ ]:
 
def build_lora_cfg(cfg: dict) -> dict:
    return {"r": cfg["lora_r"], "alpha": cfg["lora_alpha"], "dropout": cfg["lora_dropout"]}
 
 
def mark_only_lora_as_trainable(model: nn.Module) -> None:
    """
    Freeze every parameter except LoRA matrices.
    """
    for name, param in model.named_parameters():
        param.requires_grad = name.endswith(("lora_A", "lora_B"))
 
 
def merge_lora_weights(model: nn.Module) -> None:
    """
    Walk the model and call .merge() on every LoRALinear module.
    After this call the model is equivalent to a plain Transformer
    """
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module.merge()
 
 
def enable_lora(model: nn.Module) -> nn.Module:
    """
    Freeze all non-LoRA weights and report trainable parameter count.
    Call AFTER transferring pretrained weights into the LoRA model.
    """
    mark_only_lora_as_trainable(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"LoRA trainable params: {trainable:,} / {total:,}  "
          f"({100 * trainable / total:.2f} %)")
    return model
 
 
def transfer_weights_to_lora_model(pretrained: nn.Module, lora_model: nn.Module) -> nn.Module:
    """
    Copy all matching base weights from a plain model into a LoRA model.
    """
    pretrained_sd = pretrained.state_dict()
    lora_sd       = lora_model.state_dict()
 
    matched, skipped = 0, 0
    for key in lora_sd:
        if key in pretrained_sd and lora_sd[key].shape == pretrained_sd[key].shape:
            lora_sd[key] = pretrained_sd[key]
            matched += 1
        else:
            skipped += 1   # lora_A / lora_B keys leave at init values
 
    lora_model.load_state_dict(lora_sd)
    print(f"Weight transfer: {matched} matched, {skipped} skipped (LoRA deltas stay at init)")
    return lora_model

In [ ]:
def build_adapter_cfg(cfg: dict) -> dict:
    return {"bottleneck": cfg["adapter_bottleneck"], "dropout": cfg["adapter_dropout"]}
 
 
def mark_only_adapters_as_trainable(model: nn.Module) -> None:
    """
    Freeze every parameter except AdapterLayer weights.
    """
    for name, param in model.named_parameters():
        param.requires_grad = "adapter_" in name
 
 
def enable_adapters(model: nn.Module) -> nn.Module:
    """Freeze everything except adapter parameters; report trainable count."""
    mark_only_adapters_as_trainable(model)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Adapter trainable params: {trainable:,} / {total:,}  "
          f"({100 * trainable / total:.2f} %)")
    return model
 
 
def transfer_weights_to_adapter_model(pretrained: nn.Module,
                                       adapter_model: nn.Module) -> nn.Module:
    """
    Copy all matching base weights from a plain pretrained model into an
    adapter model.
    """
    pretrained_sd  = pretrained.state_dict()
    adapter_sd     = adapter_model.state_dict()
 
    matched, skipped = 0, 0
    for key in adapter_sd:
        if key in pretrained_sd and adapter_sd[key].shape == pretrained_sd[key].shape:
            adapter_sd[key] = pretrained_sd[key]
            matched += 1
        else:
            skipped += 1   # adapter down/up weights — keep zero init
 
    adapter_model.load_state_dict(adapter_sd)
    print(f"Weight transfer: {matched} matched, {skipped} skipped (adapter weights stay at init)")
    return adapter_model

In [ ]:
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
PAD_ID = tokenizer.token_to_id("<pad>")

print("BOS_ID:", BOS_ID, "EOS_ID:", EOS_ID, "PAD_ID:", PAD_ID)


def detokenize(text):
    return text.replace(" .", ".") \
               .replace(" ,", ",") \
               .replace(" !", "!") \
               .replace(" ?", "?") \
               .replace(" ' ", "'")


@torch.inference_mode()
def greedy_decode(model, src, max_len, bos_id, eos_id):
    model.eval()
    batch_size = src.size(0)
    device = src.device

    with torch.no_grad():
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()): 
            memory = model.encoder(src)

            ys = torch.full((batch_size, max_len), eos_id, dtype=torch.long, device=device)
            ys[:, 0] = bos_id
            finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

            for i in range(1, max_len):
                tgt_mask = model._causal_mask(i, device)
                out = model.decoder(ys[:, :i], memory, tgt_mask=tgt_mask)
                next_token = out[:, -1, :].argmax(dim=-1)
                ys[~finished, i] = next_token[~finished]
                finished |= (next_token == eos_id)
                if finished.all():
                    break

    return ys
"""
@torch.no_grad()
def beam_search_decode_single(model, src_single, max_len, bos_id, eos_id, beam_size=4, alpha=0.7):
    
    src_single: [1, S]
    returns: list[int]
    
    model.eval()
    memory = model.encoder(src_single)

    beams = [
        (torch.tensor([[bos_id]], device=src_single.device, dtype=torch.long), 0.0, False)
    ]  # (tokens, score, ended)

    for _ in range(max_len - 1):
        candidates = []

        for tokens, score, ended in beams:
            if ended:
                candidates.append((tokens, score, True))
                continue

            tgt_mask = model._causal_mask(tokens.size(1), src_single.device)
            out = model.decoder(tokens, memory, tgt_mask=tgt_mask)   # [1, T, V]
            log_probs = F.log_softmax(out[:, -1, :], dim=-1)         # [1, V]

            topk_log_probs, topk_ids = torch.topk(log_probs, k=beam_size, dim=-1)

            for k in range(beam_size):
                next_id = topk_ids[0, k].item()
                next_score = score + topk_log_probs[0, k].item()
                next_tokens = torch.cat(
                    [tokens, torch.tensor([[next_id]], device=src_single.device, dtype=torch.long)],
                    dim=1
                )
                candidates.append((next_tokens, next_score, next_id == eos_id))

        def norm_score(entry):
            tokens, score, _ = entry
            length = max(tokens.size(1), 1)
            return score / (length ** alpha)

        beams = sorted(candidates, key=norm_score, reverse=True)[:beam_size]

        if all(ended for _, _, ended in beams):
            break

    best_tokens = max(
        beams,
        key=lambda x: x[1] / (max(x[0].size(1), 1) ** alpha)
    )[0]

    return best_tokens.squeeze(0).tolist()
"""

@torch.no_grad()
def evaluate_with_generation(
    model,
    loader,
    criterion,
    tokenizer,
    max_len,
    decode_strategy="greedy",
    beam_size=4,
    desc="Eval"
):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    all_preds = []
    all_targets = []

    bar = tqdm.tqdm(loader, dynamic_ncols=True, leave=False, desc=desc)

    for src, tgt in bar:
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)

        # teacher-forced loss / accuracy
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        with torch.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            logits = model(src, tgt_in)
            logits_flat = logits.reshape(-1, logits.size(-1))
            tgt_flat = tgt_out.reshape(-1)
            loss = criterion(logits_flat, tgt_flat)

        total_loss += loss.item()

        preds_ids = logits.argmax(dim=-1)
        mask = tgt_out != criterion.ignore_index
        total_correct += ((preds_ids == tgt_out) & mask).sum().item()
        total_tokens += mask.sum().item()

        # real generation for BLEU
        if decode_strategy == "greedy":
            generated = greedy_decode(
                model,
                src,
                max_len=max_len,
                bos_id=BOS_ID,
                eos_id=EOS_ID
            )

            for i in range(generated.size(0)):
                pred_text = tokenizer.decode(generated[i].tolist(), skip_special_tokens=True)
                tgt_text = tokenizer.decode(tgt[i].tolist(), skip_special_tokens=True)
                all_preds.append(detokenize(pred_text))
                all_targets.append(detokenize(tgt_text))
        else:
            raise ValueError("decode_strategy must be 'greedy' or 'beam'")
        running_acc = total_correct / max(total_tokens, 1)
        running_loss = total_loss / (bar.n + 1) 
        bar.set_postfix({
            "loss": f"{running_loss:.4f}",
            "acc": f"{running_acc:.4f}"
        })

    avg_loss = total_loss / max(len(loader), 1)
    avg_acc = total_correct / max(total_tokens, 1)
    bleu = sacrebleu.corpus_bleu(all_preds, [all_targets]).score

    return avg_loss, avg_acc, bleu



def train_epoch(model, loader, optimizer, criterion, scaler, scheduler=None, grad_accum_steps=1, desc="Train"):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    optimizer.zero_grad(set_to_none=True)
    bar = tqdm.tqdm(enumerate(loader), total=len(loader), dynamic_ncols=True, leave=False, desc=desc)
    print(bar)
    for step, (src, tgt) in bar:
        src = src.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        tgt_in = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        with torch.autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            logits = model(src, tgt_in)
            logits_flat = logits.reshape(-1, logits.size(-1))
            tgt_flat = tgt_out.reshape(-1)
            raw_loss = criterion(logits_flat, tgt_flat)
            loss = raw_loss / grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            if scheduler is not None:
                scheduler.step()
                
            optimizer.zero_grad(set_to_none=True)

        total_loss += raw_loss.item()
        preds = logits_flat.argmax(dim=-1)
        mask = tgt_flat != criterion.ignore_index
        total_correct += ((preds == tgt_flat) & mask).sum().item()
        total_tokens += mask.sum().item()

        bar.set_postfix(loss=f"{raw_loss.item():.4f}")

    return total_loss / max(len(loader), 1), total_correct / max(total_tokens, 1)
    

def run_training(model, train_loader, val_loader, optimizer, criterion, scaler, scheduler, epochs, grad_accum_steps=1, stage_name="stage"):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_bleu": []}
    
    for epoch in range(epochs):
        # 1. Training Phase 
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, criterion, scaler,
            scheduler=scheduler, 
            grad_accum_steps=grad_accum_steps, 
            desc=f"{stage_name} train {epoch+1}"
        )
        
        # 2. Evaluation Phase
        val_loss, val_acc, val_bleu = evaluate_with_generation(
            model=model,
            loader=val_loader,
            criterion=criterion,
            tokenizer=tokenizer,
            max_len=CFG["max_len"],
            decode_strategy="greedy",
            desc="greedy eval")

        
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_bleu"].append(val_bleu)
        
        print(f"{stage_name} Epoch {epoch+1}:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f} | Val BLEU: {val_bleu:.2f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"  End LR:     {current_lr:.6f}")
        print("-" * 30)
        
    return history


In [ ]:

pretrain_model = Transformer(
     src_vocab_size=CFG["vocab_size"],
     tgt_vocab_size=CFG["vocab_size"],
     n_heads=CFG["n_heads"],
     d_model=CFG["d_model"],
     d_k=CFG["d_k"],
     d_v=CFG["d_v"],
     d_ff=CFG["d_ff"],
     n_layers=CFG["n_layers"],
 ).to(device)

gc.collect()
pretrain_optimizer = optim.AdamW(pretrain_model.parameters(), lr=CFG["lr_pretrain"], betas=(0.9, 0.98), eps=1e-9)
pretrain_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
pretrain_scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
total_steps = (len(pretrain_train_loader) // CFG["grad_accum_steps"]) * CFG["pretrain_epochs"]

pretrain_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    pretrain_optimizer,
    max_lr=CFG["lr_pretrain"],
    total_steps=total_steps,
    pct_start=0.1, # 10% warmup
    anneal_strategy='cos'
)

sum(p.numel() for p in pretrain_model.parameters())


In [ ]:
gc.collect()
pretrain_history = run_training(
     model=pretrain_model,
     train_loader=pretrain_train_loader,
     val_loader=pretrain_val_loader,
     optimizer=pretrain_optimizer,
     criterion=pretrain_criterion,
     scaler=pretrain_scaler,
     scheduler=pretrain_scheduler,
     epochs=CFG["pretrain_epochs"],
     grad_accum_steps=CFG["grad_accum_steps"],
     stage_name="pretrain en-fr",
 )

print("\n=== Baseline: pretrained model on Juridia val (no fine-tuning) ===")
baseline_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
baseline_loss, baseline_acc, baseline_bleu = evaluate_with_generation(
    model=pretrain_model,
    loader=finetune_val_loader,
    criterion=pretrain_criterion,
    tokenizer=tokenizer,
    max_len=CFG["max_len"],
    decode_strategy="greedy",
    desc="greedy eval"
)

print(f"  Loss: {baseline_loss:.4f} | Acc: {baseline_acc:.4f} | BLEU: {baseline_bleu:.2f}")

In [ ]:
# Save pretrained checkpoint
PRETRAIN_CKPT = "/home/ubuntu/10701/pretrained_fr_en_transformer.pt"
torch.save({
    "model_state_dict": pretrain_model.state_dict(),
    "config": CFG,
}, PRETRAIN_CKPT)
print("Saved pretrained checkpoint:", PRETRAIN_CKPT)


import wandb
import os

# Fetch the secret
wandb_key = os.getenv("WANDB_API_KEY")

# Login
wandb.login(key=wandb_key)

wandb.init(project="transformer-translation", config=CFG)

wandb.log({
    "pretrain_final_loss": pretrain_history["train_loss"][-1],
    "pretrain_final_val_loss": pretrain_history["val_loss"][-1],
})

# Save checkpoint
PRETRAIN_CKPT = "/home/ubuntu/10701/pretrained_fr_en_transformer.pt"
torch.save({
    "model_state_dict": pretrain_model.state_dict(),
    "config": CFG,
}, PRETRAIN_CKPT)

# Log artifact
artifact = wandb.Artifact(
    name="fr-en-transformer-pretrained",
    type="model",
    description="Pretrained FR→EN Transformer"
)
artifact.add_file(PRETRAIN_CKPT)
wandb.log_artifact(artifact)

print("Logged pretrained model to W&B")

In [ ]:
lora_cfg   = build_lora_cfg(CFG)
 
lora_model = Transformer(
    src_vocab_size=CFG["vocab_size"],
    tgt_vocab_size=CFG["vocab_size"],
    n_heads=CFG["n_heads"],  d_model=CFG["d_model"],
    d_k=CFG["d_k"],          d_v=CFG["d_v"],
    d_ff=CFG["d_ff"],         n_layers=CFG["n_layers"],
    lora_cfg=lora_cfg,
).to(device)
 
lora_model = transfer_weights_to_lora_model(pretrain_model, lora_model)
lora_model = enable_lora(lora_model)
 
lora_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, lora_model.parameters()),
    lr=CFG["lr_finetune"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
lora_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
lora_scaler    = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
 
total_lora_steps = (len(finetune_train_loader) // CFG["grad_accum_steps"]) * CFG["finetune_epochs"]
lora_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    lora_optimizer, max_lr=CFG["lr_finetune"],
    total_steps=total_lora_steps, pct_start=0.1, anneal_strategy='cos')
 
lora_history = run_training(
    model=lora_model,
    train_loader=finetune_train_loader,
    val_loader=finetune_val_loader,
    optimizer=lora_optimizer,
    criterion=lora_criterion,
    scaler=lora_scaler,
    scheduler=lora_scheduler,
    epochs=CFG["finetune_epochs"],
    grad_accum_steps=CFG["grad_accum_steps"],
    stage_name="finetune LoRA fr→en",
)
merge_lora_weights(lora_model)


In [ ]:
LORA_CKPT = "/home/ubuntu/10701/finetuned_lora_fr_en_transformer.pt"
torch.save({
    "model_state_dict": lora_model.state_dict(),
    "config": CFG,
}, LORA_CKPT)
print("Saved LoRA fine-tuned checkpoint:", LORA_CKPT)

In [ ]:
adapter_cfg = build_adapter_cfg(CFG)
 
adapter_model = Transformer(
    src_vocab_size=CFG["vocab_size"],
    tgt_vocab_size=CFG["vocab_size"],
    n_heads=CFG["n_heads"],  d_model=CFG["d_model"],
    d_k=CFG["d_k"],          d_v=CFG["d_v"],
    d_ff=CFG["d_ff"],         n_layers=CFG["n_layers"],
    adapter_cfg=adapter_cfg,
).to(device)
 
adapter_model = transfer_weights_to_adapter_model(pretrain_model, adapter_model)
adapter_model = enable_adapters(adapter_model)
 
adapter_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, adapter_model.parameters()),
    lr=CFG["lr_adapter"], betas=(0.9, 0.98), eps=1e-9, weight_decay=0.01)
adapter_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
adapter_scaler    = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
 
total_adapter_steps = (len(finetune_train_loader) // CFG["grad_accum_steps"]) * CFG["adapter_epochs"]
adapter_scheduler = torch.optim.lr_scheduler.OneCycleLR(
    adapter_optimizer, max_lr=CFG["lr_adapter"],
    total_steps=total_adapter_steps, pct_start=0.1, anneal_strategy='cos')
 
adapter_history = run_training(
    model=adapter_model,
    train_loader=finetune_train_loader,
    val_loader=finetune_val_loader,
    optimizer=adapter_optimizer,
    criterion=adapter_criterion,
    scaler=adapter_scaler,
    scheduler=adapter_scheduler,
    epochs=CFG["adapter_epochs"],
    grad_accum_steps=CFG["grad_accum_steps"],
    stage_name="finetune Adapter fr→en",
)
 


In [ ]:
ADAPTER_CKPT = "/home/ubuntu/10701/finetuned_adapter_fr_en_transformer.pt"
torch.save({
    "model_state_dict": adapter_model.state_dict(),
    "config": CFG,
}, ADAPTER_CKPT)
print("Saved adapter fine-tuned checkpoint:", ADAPTER_CKPT)

In [ ]:
#Code used for loading previously trained model if required


"""
import os
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRETRAIN_CKPT = "/home/ubuntu/10701/real_pretrained.pt"
LORA_CKPT     = "/home/ubuntu/10701/real_lora.pt"
ADAPTER_CKPT  = "/home/ubuntu/10701/real_adapter.pt"

BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
PAD_ID = tokenizer.token_to_id("<pad>")

def safe_detokenize(text):
    return (
        text.replace(" .", ".")
            .replace(" ,", ",")
            .replace(" !", "!")
            .replace(" ?", "?")
            .replace(" ;", ";")
            .replace(" :", ":")
            .replace(" ' ", "'")
    )

def build_plain_model(cfg):
    return Transformer(
        src_vocab_size=cfg["vocab_size"],
        tgt_vocab_size=cfg["vocab_size"],
        n_heads=cfg["n_heads"],
        d_model=cfg["d_model"],
        d_k=cfg["d_k"],
        d_v=cfg["d_v"],
        d_ff=cfg["d_ff"],
        n_layers=cfg["n_layers"],
    ).to(device)

def build_lora_model(cfg):
    return Transformer(
        src_vocab_size=cfg["vocab_size"],
        tgt_vocab_size=cfg["vocab_size"],
        n_heads=cfg["n_heads"],
        d_model=cfg["d_model"],
        d_k=cfg["d_k"],
        d_v=cfg["d_v"],
        d_ff=cfg["d_ff"],
        n_layers=cfg["n_layers"],
        lora_cfg=build_lora_cfg(cfg),
    ).to(device)

def build_adapter_model(cfg):
    return Transformer(
        src_vocab_size=cfg["vocab_size"],
        tgt_vocab_size=cfg["vocab_size"],
        n_heads=cfg["n_heads"],
        d_model=cfg["d_model"],
        d_k=cfg["d_k"],
        d_v=cfg["d_v"],
        d_ff=cfg["d_ff"],
        n_layers=cfg["n_layers"],
        adapter_cfg=build_adapter_cfg(cfg),
    ).to(device)

def load_checkpoint(path):
    assert os.path.exists(path), f"Checkpoint not found: {path}"
    ckpt = torch.load(path, map_location=device)
    cfg = ckpt["config"]
    state_dict = ckpt["model_state_dict"]
    return ckpt, cfg, state_dict

def load_pretrained(path):
    _, cfg, state_dict = load_checkpoint(path)
    model = build_plain_model(cfg)
    model.load_state_dict(state_dict)
    model.eval()
    print("Loaded pretrained from:", path)
    return model, cfg

def load_lora(path):
    _, cfg, state_dict = load_checkpoint(path)

    try:
        model = build_lora_model(cfg)
        model.load_state_dict(state_dict)
        model.eval()
        print("Loaded LoRA as LoRA-wrapped model from:", path)
        return model, cfg
    except Exception as e:
        print("LoRA-wrapped load failed, trying merged/plain load...")
        print("Reason:", e)

    model = build_plain_model(cfg)
    model.load_state_dict(state_dict)
    model.eval()
    print("Loaded LoRA as merged plain model from:", path)
    return model, cfg

def load_adapter(path):
    _, cfg, state_dict = load_checkpoint(path)
    model = build_adapter_model(cfg)
    model.load_state_dict(state_dict)
    model.eval()
    print("Loaded adapter model from:", path)
    return model, cfg

@torch.no_grad()
def compare_selected_val_indices(
    pretrain_model,
    lora_model,
    adapter_model,
    loader,
    tokenizer,
    max_len,
    indices,
    beam_size=1
):
    src_batch, tgt_batch = next(iter(loader))
    src_batch = src_batch.to(device)
    tgt_batch = tgt_batch.to(device)

    for idx in indices:
        src_text = safe_detokenize(
            tokenizer.decode(src_batch[idx].tolist(), skip_special_tokens=True)
        )
        ref_text = safe_detokenize(
            tokenizer.decode(tgt_batch[idx].tolist(), skip_special_tokens=True)
        )

        pre_pred = generate_translation(
            pretrain_model, src_text, tokenizer,
            max_len=max_len, beam_size=beam_size
        )
        lora_pred = generate_translation(
            lora_model, src_text, tokenizer,
            max_len=max_len, beam_size=beam_size
        )
        adapter_pred = generate_translation(
            adapter_model, src_text, tokenizer,
            max_len=max_len, beam_size=beam_size
        )

        print("=" * 100)
        print(f"VAL INDEX {idx}")
        print("SOURCE:     ", src_text)
        print("REFERENCE:  ", ref_text)
        print("-" * 100)
        print("PRETRAINED: ", pre_pred)
        print("LORA:       ", lora_pred)
        print("HOULSBY:    ", adapter_pred)
        print("=" * 100)
        print()

# load all 3
pretrain_model, pre_cfg = load_pretrained(PRETRAIN_CKPT)
lora_model, lora_cfg = load_lora(LORA_CKPT)
adapter_model, adapter_cfg = load_adapter(ADAPTER_CKPT)


"""